# 🎓 ĐỒ ÁN MÔN HỌC: FACE RECOGNITION (NHẬN DIỆN KHUÔN MẶT)
**Trọng tâm bài giảng:**
1. **One-shot learning problem:** Nhận diện người mới chỉ với 1 ảnh mà không cần train lại mạng nơ-ron.
2. **Siamese Network:** Mạng nơ-ron song sinh dùng chung trọng số (shared weights) để trích xuất vector đặc trưng $f(x) \in \mathbb{R}^{128}$.
3. **Cách 1 - Binary Classification:** Dự đoán cùng người hay khác người qua hiệu số $|f(x_1) - f(x_2)|$ và hàm Sigmoid.
4. **Cách 2 - Triplet Loss (FaceNet):** Huấn luyện với bộ ba Anchor (A), Positive (P), Negative (N) với Margin $\alpha$ và Hard Triplet Mining.
5. **Pre-computed Encodings:** Tính toán trước vector và so khớp bằng khoảng cách $d(x_1, x_2) \le \tau$.


## 🚀 BƯỚC 1: KIỂM TRA MÔI TRƯỜNG & GPU TRÊN COLAB
Đảm bảo bạn đã chọn GPU trên Colab: **Runtime -> Change runtime type -> T4 GPU**.


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms

# Cố định random seed để kết quả thực nghiệm có thể tái lập
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Thiết bị đang sử dụng: {device}")
if torch.cuda.is_available():
    print(f"[*] Tên GPU: {torch.cuda.get_device_name(0)}")


## 📂 BƯỚC 2: CHUẨN BỊ TẬP DỮ LIỆU KHUÔN MẶT (LFW DATASET)
Chúng ta sẽ sử dụng dataset chuẩn **LFW (Labeled Faces in the Wild)** từ `scikit-learn`.  
Lọc ra các nhân vật có ít nhất 20 ảnh để có đủ dữ liệu cho cặp Positive và Triplet.


In [ ]:
from sklearn.datasets import fetch_lfw_people

print("[*] Đang tải tập dữ liệu LFW (Labeled Faces in the Wild)...")
# Lấy những người có ít nhất 20 ảnh, ảnh màu RGB
lfw = fetch_lfw_people(min_faces_per_person=20, resize=0.8, color=True)

images = lfw.images  # shape: [N, H, W, 3] (giá trị 0-1)
labels = lfw.target
target_names = lfw.target_names

print(f"[*] Tổng số ảnh: {len(images)}")
print(f"[*] Tổng số cá nhân: {len(target_names)}")
print(f"[*] Kích thước mỗi ảnh: {images[0].shape}")

# Hiển thị thử 8 ảnh mẫu
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(target_names[labels[i]].split()[-1])
    ax.axis('off')
plt.suptitle("Một số khuôn mặt mẫu trong tập dữ liệu LFW", fontsize=14)
plt.tight_layout()
plt.show()


## 🧠 BƯỚC 3: XÂY DỰNG KIẾN TRÚC MẠNG SIAMESE (BACKBONE -> 128-D VECTOR)
Theo bài giảng:
- Dùng một mạng CNN (Backbone: ResNet-18) để trích xuất vector đặc trưng $f(x) \in \mathbb{R}^{128}$.
- Thực hiện chuẩn hóa $L_2$ norm ($||f(x)||_2 = 1$).


In [ ]:
class FaceEmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=128):
        super(FaceEmbeddingNet, self).__init__()
        # Sử dụng ResNet-18 pretrained làm backbone
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = base.fc.in_features
        base.fc = nn.Identity()
        self.backbone = base

        # Lớp đưa về vector 128 chiều
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, embedding_dim)
        )

    def forward(self, x):
        feat = self.backbone(x)
        emb = self.head(feat)
        # Chuẩn hóa L2 norm: ||f(x)||_2 = 1
        return F.normalize(emb, p=2, dim=1)

# Kiểm tra thử kích thước đầu ra
test_net = FaceEmbeddingNet().to(device)
dummy_input = torch.randn(2, 3, 160, 160).to(device)
dummy_out = test_net(dummy_input)
print(f"[*] Shape của vector đặc trưng f(x): {dummy_out.shape} (Đúng chuẩn 128 chiều)")
print(f"[*] Chuẩn L2 của vector: {torch.norm(dummy_out[0]).item():.4f} (Đã chuẩn hóa về 1.0)")


## ⚖️ BƯỚC 4: HIỆN THỰC CÁCH 1 – BINARY CLASSIFICATION (SLIDE TRANG 23–28)
Công thức trong slide:
$$\hat{y} = \sigma\left(\sum_{k=1}^{128} w_k |f(x_1)_k - f(x_2)_k| + b\right)$$
Nếu $\hat{y} \ge 0.5 \implies$ Cùng người (1), ngược lại $\implies$ Khác người (0).


In [ ]:
class SiameseBinaryNet(nn.Module):
    def __init__(self, embedding_net):
        super(SiameseBinaryNet, self).__init__()
        self.embedding_net = embedding_net
        # Lớp logistic regression trên vector sai khác |f(x1) - f(x2)|
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1)
        )

    def forward(self, x1, x2):
        f1 = self.embedding_net(x1)
        f2 = self.embedding_net(x2)
        diff = torch.abs(f1 - f2)
        logits = self.classifier(diff)
        return logits


class LFWPairDataset(Dataset):
    """Tạo các cặp ảnh (x1, x2, label in {0, 1})"""
    def __init__(self, images, labels, num_pairs=1000):
        self.images = images
        self.labels = labels
        self.num_pairs = num_pairs
        self.label_to_indices = {}
        for idx, lbl in enumerate(labels):
            self.label_to_indices.setdefault(lbl, []).append(idx)
        self.eligible_labels = [lbl for lbl, idxs in self.label_to_indices.items() if len(idxs) >= 2]
        self.all_labels = list(self.label_to_indices.keys())

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((160, 160)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return self.num_pairs

    def __getitem__(self, idx):
        is_same = (idx % 2 == 0)
        if is_same:
            lbl = random.choice(self.eligible_labels)
            idx1, idx2 = random.sample(self.label_to_indices[lbl], 2)
            y = 1.0
        else:
            lbl1, lbl2 = random.sample(self.all_labels, 2)
            idx1 = random.choice(self.label_to_indices[lbl1])
            idx2 = random.choice(self.label_to_indices[lbl2])
            y = 0.0

        # images có dạng [H, W, 3] với giá trị float 0-1 -> chuyển uint8 0-255
        img1 = (self.images[idx1] * 255).astype(np.uint8)
        img2 = (self.images[idx2] * 255).astype(np.uint8)

        t1 = self.transform(img1)
        t2 = self.transform(img2)
        return t1, t2, torch.tensor(y, dtype=torch.float32)

# Khởi tạo DataLoader
pair_dataset = LFWPairDataset(images, labels, num_pairs=1200)
pair_loader = DataLoader(pair_dataset, batch_size=32, shuffle=True)
print(f"[*] Đã tạo PairDataset với {len(pair_dataset)} cặp ảnh.")


### Huấn luyện Mô hình Binary Classification


In [ ]:
binary_emb_net = FaceEmbeddingNet().to(device)
binary_model = SiameseBinaryNet(binary_emb_net).to(device)

criterion_bce = nn.BCEWithLogitsLoss()
optimizer_bce = torch.optim.Adam(binary_model.parameters(), lr=1e-4)

print("[*] Bắt đầu huấn luyện mô hình Siamese Binary Classification (5 epochs)...")
binary_model.train()
binary_losses = []
binary_accs = []

for epoch in range(1, 6):
    total_loss, correct, total = 0.0, 0, 0
    for x1, x2, y in pair_loader:
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)
        optimizer_bce.zero_grad()
        logits = binary_model(x1, x2).squeeze(1)
        loss = criterion_bce(logits, y)
        loss.backward()
        optimizer_bce.step()

        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)
        total_loss += loss.item() * y.size(0)

    ep_loss = total_loss / total
    ep_acc = correct / total
    binary_losses.append(ep_loss)
    binary_accs.append(ep_acc)
    print(f"Epoch [{epoch}/5] - Loss: {ep_loss:.4f} | Accuracy: {ep_acc:.2%}")

print("[✓] Huấn luyện hoàn thành Cách 1!")


## 🎯 BƯỚC 5: HIỆN THỰC CÁCH 2 – TRIPLET LOSS & HARD MINING (SLIDE TRANG 13–22)
Công thức Triplet Loss:
$$\mathcal{L}(A, P, N) = \max\left(\|f(A) - f(P)\|^2 - \|f(A) - f(N)\|^2 + \alpha, 0\right)$$
- $\alpha$: Margin (khoảng đệm), thường chọn $0.2$.
- **Hard Triplet Mining:** Chọn mẫu Negative khó sao cho $d(A, N) \approx d(A, P)$ để ép mạng nơ-ron học triệt để hơn.


In [ ]:
class TripletLoss(nn.Module):
    def __init__(self, alpha=0.2):
        super(TripletLoss, self).__init__()
        self.alpha = alpha

    def forward(self, anchor, positive, negative):
        # Bình phương khoảng cách Euclidean
        d_ap = F.pairwise_distance(anchor, positive, p=2).pow(2)
        d_an = F.pairwise_distance(anchor, negative, p=2).pow(2)
        loss = F.relu(d_ap - d_an + self.alpha)
        return loss.mean(), d_ap.mean().item(), d_an.mean().item()


class LFWTripletDataset(Dataset):
    """Tạo các bộ ba (Anchor, Positive, Negative)"""
    def __init__(self, images, labels, num_triplets=1200):
        self.images = images
        self.labels = labels
        self.num_triplets = num_triplets
        self.label_to_indices = {}
        for idx, lbl in enumerate(labels):
            self.label_to_indices.setdefault(lbl, []).append(idx)
        self.eligible_labels = [lbl for lbl, idxs in self.label_to_indices.items() if len(idxs) >= 2]
        self.all_labels = list(self.label_to_indices.keys())

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((160, 160)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return self.num_triplets

    def __getitem__(self, idx):
        # 1. Chọn người cho Anchor và Positive
        lbl_p = random.choice(self.eligible_labels)
        idx_a, idx_p = random.sample(self.label_to_indices[lbl_p], 2)

        # 2. Chọn người khác cho Negative
        lbl_n = random.choice([l for l in self.all_labels if l != lbl_p])
        idx_n = random.choice(self.label_to_indices[lbl_n])

        img_a = self.transform((self.images[idx_a] * 255).astype(np.uint8))
        img_p = self.transform((self.images[idx_p] * 255).astype(np.uint8))
        img_n = self.transform((self.images[idx_n] * 255).astype(np.uint8))
        return img_a, img_p, img_n

triplet_dataset = LFWTripletDataset(images, labels, num_triplets=1200)
triplet_loader = DataLoader(triplet_dataset, batch_size=32, shuffle=True)
print(f"[*] Đã tạo TripletDataset với {len(triplet_dataset)} bộ ba.")


### Huấn luyện Mô hình với Triplet Loss
Quan sát sự thay đổi của $d(A, P)$ (khoảng cách cùng người) và $d(A, N)$ (khoảng cách khác người).  
Mục tiêu: $d(A, P)$ giảm dần về $0$, $d(A, N)$ tăng lớn hơn $d(A, P) + \alpha$.


In [ ]:
triplet_model = FaceEmbeddingNet().to(device)
criterion_triplet = TripletLoss(alpha=0.2)
optimizer_triplet = torch.optim.Adam(triplet_model.parameters(), lr=1e-4)

print("[*] Bắt đầu huấn luyện mô hình Triplet Loss (5 epochs)...")
triplet_model.train()
triplet_losses = []
d_ap_history = []
d_an_history = []

for epoch in range(1, 6):
    total_loss, total_ap, total_an, count = 0.0, 0.0, 0.0, 0
    for a, p, n in triplet_loader:
        a, p, n = a.to(device), p.to(device), n.to(device)
        optimizer_triplet.zero_grad()

        f_a = triplet_model(a)
        f_p = triplet_model(p)
        f_n = triplet_model(n)

        loss, ap, an = criterion_triplet(f_a, f_p, f_n)
        loss.backward()
        optimizer_triplet.step()

        b_size = a.size(0)
        total_loss += loss.item() * b_size
        total_ap += ap * b_size
        total_an += an * b_size
        count += b_size

    ep_loss = total_loss / count
    ep_ap = total_ap / count
    ep_an = total_an / count
    triplet_losses.append(ep_loss)
    d_ap_history.append(ep_ap)
    d_an_history.append(ep_an)
    print(f"Epoch [{epoch}/5] - Triplet Loss: {ep_loss:.4f} | d(A,P): {ep_ap:.3f} | d(A,N): {ep_an:.3f}")

print("[✓] Huấn luyện hoàn thành Cách 2!")


## 📊 BƯỚC 6: TRỰC QUAN HÓA KẾT QUẢ HUẤN LUYỆN
Vẽ biểu đồ so sánh:
1. Độ suy giảm của Triplet Loss.
2. Sự tách biệt giữa khoảng cách cùng người $d(A, P)$ và khoảng cách khác người $d(A, N)$.


In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, 6), triplet_losses, marker='o', color='purple', label='Triplet Loss')
plt.title("Đồ thị hàm mất mát Triplet Loss theo Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, 6), d_ap_history, marker='o', color='green', label='Cùng người: d(A, P)')
plt.plot(range(1, 6), d_an_history, marker='s', color='red', label='Khác người: d(A, N)')
plt.title("Khoảng cách Euclidean giữa các cặp")
plt.xlabel("Epoch")
plt.ylabel("Khoảng cách bình phương d")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


## 🗺️ BƯỚC 7: TRỰC QUAN HÓA KHÔNG GIAN EMBEDDING BẰNG T-SNE
Chiếu vector 128 chiều xuống không gian 2D để xem mạng nơ-ron gom cụm các khuôn mặt của cùng một người như thế nào!


In [ ]:
from sklearn.manifold import TSNE

# Chọn 5 người có nhiều ảnh nhất để vẽ t-SNE
top_5_labels = [lbl for lbl, _ in sorted([(l, len(idxs)) for l, idxs in triplet_dataset.label_to_indices.items()], key=lambda x: x[1], reverse=True)[:5]]

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

triplet_model.eval()
emb_list, lbl_list = [], []

with torch.no_grad():
    for lbl in top_5_labels:
        for idx in triplet_dataset.label_to_indices[lbl]:
            img = eval_transform((images[idx] * 255).astype(np.uint8)).unsqueeze(0).to(device)
            emb = triplet_model(img).cpu().numpy().flatten()
            emb_list.append(emb)
            lbl_list.append(target_names[lbl])

emb_array = np.array(emb_list)
tsne = TSNE(n_components=2, random_state=42, perplexity=10)
emb_2d = tsne.fit_transform(emb_array)

plt.figure(figsize=(9, 7))
for name in set(lbl_list):
    mask = [lbl == name for lbl in lbl_list]
    plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=name, s=60, alpha=0.85)

plt.title("Trực quan hóa không gian đặc trưng 128-D bằng t-SNE (Top 5 người)", fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 🔑 BƯỚC 8: ONE-SHOT LEARNING & PRE-COMPUTED ENCODINGS (DEMO THỰC TẾ)
Bám sát slide:
1. Đăng ký người mới vào Database: Chỉ lấy đúng **1 ảnh duy nhất** tính vector 128 chiều $f(x)$ lưu vào `database`.
2. Nhận diện ảnh mới: Tính khoảng cách $d(x, x_{\text{db}}) = \|f(x) - f(x_{\text{db}})\|^2$.  
   Nếu $\min(d) \le \tau = 0.6 \implies$ Nhận diện đúng danh tính!


In [ ]:
# Tạo database lưu vector tính sẵn (Pre-computed Encodings) cho 4 người đầu tiên
database = {}
query_faces = []

with torch.no_grad():
    for i in range(4):
        person_name = target_names[top_5_labels[i]]
        # Lấy ảnh số 0 làm ảnh đăng ký (Enrollment)
        enroll_idx = triplet_dataset.label_to_indices[top_5_labels[i]][0]
        enroll_tensor = eval_transform((images[enroll_idx] * 255).astype(np.uint8)).unsqueeze(0).to(device)
        enroll_emb = triplet_model(enroll_tensor).cpu().numpy().flatten()
        database[person_name] = enroll_emb

        # Lấy ảnh số 1 làm ảnh kiểm tra (Query Test)
        test_idx = triplet_dataset.label_to_indices[top_5_labels[i]][1]
        query_faces.append((person_name, images[test_idx]))

print(f"[*] Đã đăng ký One-Shot cho {len(database)} người vào Database:")
for name in database.keys():
    print(f"   [+] {name}")

# Thử nghiệm nhận diện trên các ảnh Test
threshold_tau = 0.65

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
with torch.no_grad():
    for i, (true_name, test_img) in enumerate(query_faces):
        test_tensor = eval_transform((test_img * 255).astype(np.uint8)).unsqueeze(0).to(device)
        test_emb = triplet_model(test_tensor).cpu().numpy().flatten()

        # Tính khoảng cách Euclidean tới từng người trong database
        min_dist = float('inf')
        predicted_name = "Unknown"
        for db_name, db_emb in database.items():
            dist = np.sum((test_emb - db_emb) ** 2)
            if dist < min_dist:
                min_dist = dist
                if dist <= threshold_tau:
                    predicted_name = db_name

        ax = axes[i]
        ax.imshow(test_img)
        title_color = 'green' if predicted_name == true_name else 'red'
        ax.set_title(f"Truth: {true_name.split()[-1]}\nPred: {predicted_name.split()[-1]} (d={min_dist:.2f})", color=title_color)
        ax.axis('off')

plt.suptitle(f"Kết quả One-Shot Face Recognition (Ngưỡng tau = {threshold_tau})", fontsize=14)
plt.tight_layout()
plt.show()


## 💾 BƯỚC 9: LƯU TRỌNG SỐ MÔ HÌNH ĐỂ TẢI VỀ MÁY CHẠY WEBCAM
Tải file trọng số này về để chạy với script `recognize.py` trên máy tính cá nhân có webcam!


In [ ]:
import pickle

# Lưu trọng số mô hình Triplet Embedding
torch.save(triplet_model.state_dict(), "siamese_triplet.pth")
print("[✓] Đã lưu file trọng số: siamese_triplet.pth")

# Lưu database vector
with open("face_db.pkl", "wb") as f:
    pickle.dump(database, f)
print("[✓] Đã lưu database: face_db.pkl")

# Lệnh tải file về máy (nếu chạy trên Google Colab)
try:
    from google.colab import files
    files.download('siamese_triplet.pth')
    files.download('face_db.pkl')
    print("[*] Đang bắt đầu tải file về máy tính của bạn...")
except ImportError:
    print("[*] Chạy offline, các file đã nằm sẵn trong thư mục làm việc.")
